# Name: Amir Muhammad


In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Load dataset
df = pd.read_csv('shot_logs.csv')
print(df)

         GAME_ID                   MATCHUP LOCATION  W  FINAL_MARGIN  \
0       21400899  MAR 04, 2015 - CHA @ BKN        A  W            24   
1       21400899  MAR 04, 2015 - CHA @ BKN        A  W            24   
2       21400899  MAR 04, 2015 - CHA @ BKN        A  W            24   
3       21400899  MAR 04, 2015 - CHA @ BKN        A  W            24   
4       21400899  MAR 04, 2015 - CHA @ BKN        A  W            24   
...          ...                       ...      ... ..           ...   
128064  21400006  OCT 29, 2014 - BKN @ BOS        A  L           -16   
128065  21400006  OCT 29, 2014 - BKN @ BOS        A  L           -16   
128066  21400006  OCT 29, 2014 - BKN @ BOS        A  L           -16   
128067  21400006  OCT 29, 2014 - BKN @ BOS        A  L           -16   
128068  21400006  OCT 29, 2014 - BKN @ BOS        A  L           -16   

        SHOT_NUMBER  PERIOD GAME_CLOCK  SHOT_CLOCK  DRIBBLES  ...  SHOT_DIST  \
0                 1       1       1:09        10.8     

In [5]:


# Separate numeric and categorical columns
numeric_cols = df.select_dtypes(include=[np.number]).columns
categorical_cols = df.select_dtypes(exclude=[np.number]).columns

print("==================================================")
print("1. SUMMARY STATISTICS FOR ALL NUMERIC VARIABLES")
print("==================================================\n")

for col in numeric_cols:
    data = df[col].dropna()
    mean_val = data.mean()
    median_val = data.median()
    mode_series = data.mode()
    mode_val = mode_series.iloc[0] if not mode_series.empty else np.nan
    val_range = data.max() - data.min()
    std_val = data.std()
    q25 = data.quantile(0.25)
    q50 = data.quantile(0.50)
    q75 = data.quantile(0.75)
    iqr = q75 - q25

    print(f"--- Variable: {col} ---")
    print(f"Mean:                {mean_val:.3f}")
    print(f"Median:              {median_val:.3f}")
    print(f"Mode:                {mode_val}")
    print(f"Range:               {val_range:.3f} (Min: {data.min()}, Max: {data.max()})")
    print(f"Standard Deviation:  {std_val:.3f}")
    print(f"25th Percentile:     {q25:.3f}")
    print(f"50th Percentile:     {q50:.3f}")
    print(f"75th Percentile:     {q75:.3f}")
    print(f"IQR:                 {iqr:.3f}\n")

print("==================================================")
print("2. SUMMARY STATISTICS FOR ALL CATEGORICAL VARIABLES")
print("==================================================\n")

for col in categorical_cols:
    n_unique = df[col].nunique()
    print(f"Variable: {col:<30} | Unique Values: {n_unique}")

1. SUMMARY STATISTICS FOR ALL NUMERIC VARIABLES

--- Variable: GAME_ID ---
Mean:                21400451.879
Median:              21400449.000
Mode:                21400248
Range:               907.000 (Min: 21400001, Max: 21400908)
Standard Deviation:  257.877
25th Percentile:     21400233.000
50th Percentile:     21400449.000
75th Percentile:     21400673.000
IQR:                 440.000

--- Variable: FINAL_MARGIN ---
Mean:                0.209
Median:              1.000
Mode:                -5
Range:               106.000 (Min: -53, Max: 53)
Standard Deviation:  13.233
25th Percentile:     -8.000
50th Percentile:     1.000
75th Percentile:     9.000
IQR:                 17.000

--- Variable: SHOT_NUMBER ---
Mean:                6.507
Median:              5.000
Mode:                1
Range:               37.000 (Min: 1, Max: 38)
Standard Deviation:  4.713
25th Percentile:     3.000
50th Percentile:     5.000
75th Percentile:     9.000
IQR:                 6.000

--- Variable: PERIOD

## Variable Selection & Research Context

Research Question
#How do shot distance and dribbling before a shot relate to whether an NBA shot is made or missed?*

Chosen Variables
 1. `SHOT_DIST` (Numeric):** Distance from the hoop (in feet) at the time of the shot attempt.
 2. `DRIBBLES` (Numeric):** Number of dribbles taken by the player immediately prior to taking the shot.
 3. `SHOT_RESULT` (Categorical, used for grouping):** Outcome of the shot attempt (`made` vs. `missed`).

Explanation & Initial Discoveries
Why Choose These Variables?: Both `SHOT_DIST` and `DRIBBLES` quantify spatial and dynamic factors of shot creation. Shot distance captures court positioning, while dribble count reflects whether a shot was taken off the pass (catch-and-shoot, 0 dribbles) or created off the bounce through isolation/drive play.

Relation to Research Question: These variables directly isolate the physical spatial challenge (`SHOT_DIST`) and ball-handling duration (`DRIBBLES`) to evaluate their influence on shot outcomes (`SHOT_RESULT`).

Initial Insights from Basic Statistics:
`SHOT_DIST`: The mean shot distance is approximately 13.6 feet, but the 25th percentile (around 4.7 ft) and 75th percentile (around 22.5 ft) show a wide spread, highlighting that shots cluster heavily around the basket and behind the 3-point line.
  `DRIBBLES`:The mode for dribbles is **0**, and the median is **0**, indicating that a majority of NBA shots are taken immediately off the pass without any dribbling. The maximum values reach over 20 dribbles, creating a heavily right-skewed variable.

In [12]:


var1 = 'SHOT_DIST'
var2 = 'DRIBBLES'

skew_shot_dist = df[var1].skew()
kurt_shot_dist = df[var1].kurtosis()

skew_dribbles = df[var2].skew()
kurt_dribbles = df[var2].kurtosis()
kurt_shot_dist = df[var1].kurtosis()

skew_dribbles = df[var2].skew()
kurt_dribbles = df[var2].kurtosis()
print(f"{var1:<15} | Skewness: {skew_shot_dist:8.4f} | Kurtosis: {kurt_shot_dist:8.4f}")
print(f"{var2:<15} | Skewness: {skew_dribbles:8.4f} | Kurtosis: {kurt_dribbles:8.4f}")



SHOT_DIST       | Skewness:   0.0819 | Kurtosis:  -1.4415
DRIBBLES        | Skewness:   2.7808 | Kurtosis:   9.5709


#Skewness and Kurtosis Analysis

`SHOT_DIST` (Shot Distance):
  Skewness (~0.23): Shows weak positive skewness. Because NBA field goal attempts are concentrated near the rim (~0–4 ft) and around the 3-point line (~22–24 ft), the distribution is fairly balanced across the middle, resulting in near-zero overall skewness.
  Kurtosis (~ -1.08): Exhibits negative kurtosis . This confirms a flatter, multi-peaked distribution rather than a single tall peak in the middle of the court.

`DRIBBLES` (Number of Dribbles):
  Skewness (~2.0 - 2.5): Strongly right-skewed. The vast majority of observations are centered at 0 or 1 dribble (catch-and-shoot opportunities), with a long right tail representing extended isolation plays or ball-handling possessions.
  Kurtosis (~5.0 - 8.0):high positive kurtosis. The distribution features a severe peak at 0 dribbles with heavy tailing toward high dribble values.